In [6]:
"""
smolagents + SQLite Long-Term Memory Pattern
===========================================
Uses a local SQLite file (`agent_memory.db`) to persist knowledge across tasks
without inflating the prompt's context window.
"""

import json
import os
import sqlite3
from typing import Optional
from smolagents import CodeAgent, OpenAIServerModel, tool

# 1. Database Initialization
DB_FILE = "agent_memory.db"

def init_db():
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            CREATE TABLE IF NOT EXISTS memories (
                key TEXT PRIMARY KEY,
                category TEXT,
                value TEXT,
                status TEXT,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
            """
        )
        conn.commit()

init_db()

# 3. Model & Agent Setup
OMLX_API_KEY = "sk-omlx-gWACPPkjsP67zAEgJ3GReJ6q"
OMLX_BASE_URL = "http://localhost:8000/v1" 
MODEL_ID = "Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit"

model = OpenAIServerModel(
    model_id=MODEL_ID,
    api_base=OMLX_BASE_URL,
    api_key=OMLX_API_KEY,
    temperature=0.1,
)

In [7]:
# 2. Local Memory Tools
@tool
def store_memory(key: str, category: str, value: str, status: str = "active") -> str:
    """Inserts or updates a persistent memory record in the database.

    Args:
        key: Unique slug or identifier (e.g., 'user_target_weight', 'project_alpha').
        category: Broad classification (e.g., 'fitness', 'project', 'finance').
        value: The detailed information/content to store.
        status: The state of this entry (e.g., 'active', 'pending', 'completed').

    Returns:
        str: Status message indicating whether the record was saved.
    """
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute(
            """
            INSERT INTO memories (key, category, value, status, updated_at)
            VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP)
            ON CONFLICT(key) DO UPDATE SET
                category=excluded.category,
                value=excluded.value,
                status=excluded.status,
                updated_at=CURRENT_TIMESTAMP
            """,
            (key, category, value, status),
        )
        conn.commit()
    return f"Successfully saved memory entry for key '{key}'."


@tool
def fetch_memories(category: Optional[str] = None, status: Optional[str] = None) -> str:
    """Retrieves stored memories from SQLite, optionally filtered by category or status.

    Args:
        category: Filter by specific category (optional).
        status: Filter by specific status (optional).

    Returns:
        str: JSON list of matching memory records.
    """
    query = "SELECT key, category, value, status, updated_at FROM memories WHERE 1=1"
    params = []
    
    if category:
        query += " AND category = ?"
        params.append(category)
    if status:
        query += " AND status = ?"
        params.append(status)

    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(query, params)
        rows = [dict(row) for row in cursor.fetchall()]

    if not rows:
        return "No memory records found matching criteria."
    return json.dumps(rows, indent=2)


@tool
def update_memory_status(key: str, new_status: str, appended_note: Optional[str] = None) -> str:
    """Modifies the status and optionally appends a note to an existing memory record.

    Args:
        key: The unique identifier of the entry to modify.
        new_status: The new status value (e.g., 'completed', 'blocked', 'archived').
        appended_note: Optional extra context to append to the existing value.

    Returns:
        str: Status message reporting the update result.
    """
    with sqlite3.connect(DB_FILE) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT value FROM memories WHERE key = ?", (key,))
        row = cursor.fetchone()
        if not row:
            return f"Error: No entry found for key '{key}'."
        
        current_value = row[0]
        updated_value = (
            f"{current_value} | Note: {appended_note}" if appended_note else current_value
        )

        cursor.execute(
            """
            UPDATE memories 
            SET status = ?, value = ?, updated_at = CURRENT_TIMESTAMP
            WHERE key = ?
            """,
            (new_status, updated_value, key),
        )
        conn.commit()
    return f"Updated key '{key}' to status '{new_status}'."

In [8]:
memory_agent = CodeAgent(
    tools=[store_memory, fetch_memories, update_memory_status],
    model=model,
    add_base_tools=False,
    additional_authorized_imports=["json"],
    max_steps=5,
    verbosity_level=1,
)

In [9]:
# --- TASK 1: Initialize knowledge into the DB ---
print("=" * 60)
print("TASK 1: Persist initial tasks and context into SQLite")
print("=" * 60)
task_1 = """
We have 3 initiatives to register into the system:
1. Key 'db_migration': category 'infrastructure', value 'Migrate Postgres cluster to v16', status 'pending'.
2. Key 'auth_refactor': category 'backend', value 'Implement OAuth2 PKCE flow', status 'in_progress'.
3. Key 'benchmarks': category 'analytics', value 'Profile local MLX throughput for 9B models', status 'pending'.

Use `store_memory` to save all three into SQLite. Then provide a one-line confirmation.
"""
res_1 = memory_agent.run(task_1)
print("\n[Task 1 Result]:", res_1)

TASK 1: Persist initial tasks and context into SQLite


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ We have 3 initiatives to register into the system:                                                              │
│ 1. Key 'db_migration': category 'infrastructure', value 'Migrate Postgres cluster to v16', status 'pending'.    │
│ 2. Key 'auth_refactor': category 'backend', value 'Implement OAuth2 PKCE flow', status 'in_progress'.           │
│ 3. Key 'benchmarks': category 'analytics', value 'Profile local MLX throughput for 9B models', status           │
│ 'pending'.                                                                                                      │
│                                                                                                                 │
│ Use `store_memory` to save all three into SQLite. Then provide a one-line confirmation.                         │
│                                                                                                                 │
╰─ OpenAIModel - Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit ───────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Store all three initiatives using store_memory                                                                 
  store_memory('db_migration', 'infrastructure', 'Migrate Postgres cluster to v16', 'pending')                     
  store_memory('auth_refactor', 'backend', 'Implement OAuth2 PKCE flow', 'in_progress')                            
  store_memory('benchmarks', 'analytics', 'Profile local MLX throughput for 9B models', 'pending')                 
                                                                                                                   
  # Print one-line confirmation                                                                                    
  print("Successfully stored 3 initiatives: db_migration (infrastructure), auth_refactor (backend), benchmarks     
  (analytics)")                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Successfully stored 3 initiatives: db_migration (infrastructure), auth_refactor (backend), benchmarks (analytics)

Out: None

[Step 1: Duration 9.50 seconds| Input tokens: 2,532 | Output tokens: 141]

... (16 outputs left out)

[Step 5: Duration 2.21 seconds| Input tokens: 15,926 | Output tokens: 366]

Reached max steps.

[Step 6: Duration 11.32 seconds| Input tokens: 17,898 | Output tokens: 640]


[Task 1 Result]: <code>
# Store all three initiatives using store_memory
store_memory('db_migration', 'infrastructure', 'Migrate Postgres cluster to v16', 'pending')
store_memory('auth_refactor', 'backend', 'Implement OAuth2 PKCE flow', 'in_progress')
store_memory('benchmarks', 'analytics', 'Profile local MLX throughput for 9B models', 'pending')

# Print one-line confirmation
print("Successfully stored 3 initiatives: db_migration (infrastructure), auth_refactor (backend), benchmarks (analytics)")</code>
Calling tools:
[{'id': 'call_6', 'type': 'function', 'function': {'name': 'python_interpreter', 'arguments': '# Store all three initiatives using store_memory\nstore_memory(\'db_migration\', \'infrastructure\', \'Migrate Postgres cluster to v16\', \'pending\')\nstore_memory(\'auth_refactor\', \'backend\', \'Implement OAuth2 PKCE flow\', \'in_progress\')\nstore_memory(\'benchmarks\', \'analytics\', \'Profile local MLX throughput for 9B models\', \'pending\')\n\n# Print one-line confirm

In [10]:
# --- TASK 2: Read, inspect, and update in a separate context ---
print("\n" + "=" * 60)
print("TASK 2: Query DB, review status, and update a task")
print("=" * 60)
task_2 = """
1. Fetch all memories with status 'pending' using `fetch_memories`.
2. Check if 'benchmarks' is among them.
3. If found, use `update_memory_status` to change its status to 'completed' with the note 'Achieved 42 tokens/sec'.
4. Fetch the updated record to verify and state the final result.
"""
res_2 = memory_agent.run(task_2)
print("\n[Task 2 Result]:", res_2)


TASK 2: Query DB, review status, and update a task


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 1. Fetch all memories with status 'pending' using `fetch_memories`.                                             │
│ 2. Check if 'benchmarks' is among them.                                                                         │
│ 3. If found, use `update_memory_status` to change its status to 'completed' with the note 'Achieved 42          │
│ tokens/sec'.                                                                                                    │
│ 4. Fetch the updated record to verify and state the final result.                                               │
│                                                                                                                 │
╰─ OpenAIModel - Qwen3.5-9B-Claude-4.6-HighIQ-INSTRUCT-HERETIC-UNCENSORED-mlx-4Bit ───────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Step 1: Fetch all memories with status 'pending'                                                               
  pending_memories = fetch_memories(category=None, status='pending')                                               
  print("Pending memories:", pending_memories)                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Pending memories: [
  {
    "key": "db_migration",
    "category": "infrastructure",
    "value": "Migrate Postgres cluster to v16",
    "status": "pending",
    "updated_at": "2026-11-01 11:23:34"
  },
  {
    "key": "benchmarks",
    "category": "analytics",
    "value": "Profile local MLX throughput for 9B models",
    "status": "pending",
    "updated_at": "2026-11-01 11:23:34"
  }
]

Out: None

[Step 1: Duration 3.02 seconds| Input tokens: 2,492 | Output tokens: 77]

... (15 outputs left out)

[Step 5: Duration 7.16 seconds| Input tokens: 15,759 | Output tokens: 647]

Reached max steps.

[Step 6: Duration 12.78 seconds| Input tokens: 18,007 | Output tokens: 952]


[Task 2 Result]: ## Final Answer

The task has been **successfully completed**. Here's the summary:

### Step-by-Step Execution:

| Step | Action | Result |
|------|--------|--------|
| 1 | Fetched all 'pending' memories | Found 2 memories: `db_migration` and `benchmarks` |
| 2 | Checked for 'benchmarks' | **Found** - key "benchmarks" exists with status "pending" |
| 3 | Updated status to 'completed' | Successfully updated with note "Achieved 42 tokens/sec" |
| 4 | Verified the update | Confirmed in 'completed' status |

### Final Result:

```
Updated Benchmarks Record:
┌─────────────────────────────────────────────────────────────┐
│ Key:      benchmarks                                       │
│ Category: analytics                                         │
│ Value:    Profile local MLX throughput for 9B models        │
│           | Note: Achieved 42 tokens/sec                    │
│ Status:   completed                                         │
│ Updated:  2026-11-01 11:24:32        